# 08D_Extract_Technical_Skills_v2

Extract ONLY technology skills from LinkedIn skill universe.

Output:
- technical_skill_master.csv

In [16]:
import pandas as pd
import re
from rapidfuzz import process, fuzz

In [17]:
skills = pd.read_csv('../Generated Datasets/linkedin_skill_universe_1000.csv')
skills['skill'] = skills['skill'].astype(str).str.lower().str.strip()

## Technology Taxonomy

In [18]:
TECH = {

    'Programming': [
        'python','java','javascript','typescript','php',
        'kotlin','swift','golang','rust',
        'c','c++','c#','r','matlab',
        'programming','coding',
        'object oriented programming',
        'oop','react',
        'angular',
        'vue',
        'node.js',
        'nodejs',
        'next.js',
        'nextjs',
        'express.js',
        'express',
        'flask',
        'django'
    ],

    'AI_ML': [
        'machine learning',
        'deep learning',
        'computer vision',
        'natural language processing',
        'nlp',
        'llm',
        'rag',
        'langchain',
        'tensorflow',
        'pytorch',
        'scikit-learn',
        'artificial intelligence',
        'data science',
        'predictive modeling',
        'feature engineering',
        'neural networks',
        'generative ai',
        'chatgpt',
        'prompt engineering'
    ],

    'Data Analytics': [
        'data analysis',
        'data analytics',
        'statistics',
        'statistical analysis',
        'business analytics',
        'tableau',
        'power bi',
        'powerbi',
        'excel',
        'microsoft excel',
        'ms excel',
        'business intelligence',
        'data visualization',
        'dashboard development',
        'reporting'
    ],

    'Database': [
        'sql',
        'mysql',
        'postgresql',
        'mongodb',
        'oracle',
        'snowflake',
        'nosql',
        'database',
        'database management',
        'data modeling',
        'postgres',
        'postgresql',
        'sql server',
        'database management',
        'data modeling'
    ],

    'Cloud': [
        'aws',
        'azure',
        'gcp',
        'google cloud',
        'cloud computing',
        'amazon web services',
        'microsoft azure',
        'cloud architecture',
        'serverless',
        'amazon web services',
        'microsoft azure',
        'cloud architecture',
        'serverless'
    ],

    'DevOps': [
        'docker',
        'kubernetes',
        'jenkins',
        'terraform',
        'gitlab',
        'github',
        'git',
        'github actions',
        'ci/cd',
        'devops',
        'linux',
        'unix',
        'github',
        'github actions',
        'terraform',
        'ci/cd',
        'continuous integration',
        'continuous deployment'
    ],

    'Cybersecurity': [
        'cybersecurity',
        'network security',
        'cloud security',
        'penetration testing',
        'ethical hacking',
        'information security',
        'information security',
        'security operations',
        'vulnerability assessment',
        'ethical hacking'
    ],

    'Software Engineering': [
        'software engineering',
        'software development',
        'agile development',
        'agile',
        'system design',
        'api',
        'apis',
        'rest api',
        'microservices',
        'backend development',
        'frontend development',
        'full stack development',
        'testing',
        'unit testing',
        'software testing',
        'quality assurance',
        'automation'
    ],

    'Data Engineering': [
        'data engineering',
        'etl',
        'data warehouse',
        'big data',
        'data pipeline',
        'data architecture',
        'data governance',
        'data pipeline',
        'data pipelines',
        'data architecture',
        'data governance',
        'apache spark',
        'hadoop'
    ],

    'Tools_Platforms': [
        'jira',
        'confluence',
        'sharepoint',
        'informatica',
        'information technology'
    ],

    'Engineering_Systems': [
        'embedded systems',
        'robotics',
        'electronics',
        'electrical engineering',
        'mechanical engineering',
        'systems engineering',
        'computer engineering'
    ]
}

In [19]:
taxonomy = {}
all_terms = []

for subcat, terms in TECH.items():
    for term in terms:
        taxonomy[term] = subcat
        all_terms.append(term)

## Exact + Regex + Fuzzy Matching

In [20]:
results = []
freq_col = [c for c in skills.columns if c != 'skill'][0]

for _, row in skills.iterrows():

    skill = row['skill']
    freq = row[freq_col]

    matched = False

    if skill in taxonomy:
        results.append([
            skill,freq,taxonomy[skill],1.00
        ])
        continue

    for term, subcat in taxonomy.items():
        pattern = r'\b' + re.escape(term) + r'\b'

        if re.search(pattern, skill):
            results.append([
                skill,freq,subcat,0.95
            ])
            matched = True
            break

    if matched:
        continue

    fuzzy = process.extractOne(
        skill,
        all_terms,
        scorer=fuzz.token_sort_ratio
    )

    if fuzzy and fuzzy[1] >= 90:
        results.append([
            skill,
            freq,
            taxonomy[fuzzy[0]],
            round(fuzzy[1]/100,2)
        ])

In [21]:
technical_df = pd.DataFrame(
    results,
    columns=[
        'skill',
        'linkedin_frequency',
        'sub_category',
        'confidence'
    ]
)

technical_df = technical_df.drop_duplicates('skill')

technical_df = technical_df.sort_values(
    'linkedin_frequency',
    ascending=False
)

technical_df.to_csv(
    '../Generated Datasets/technical_skill_master.csv',
    index=False
)

print('Technical Skills:', len(technical_df))
print(technical_df['sub_category'].value_counts())

Technical Skills: 128
sub_category
Software Engineering    24
Data Analytics          20
Programming             20
Database                13
DevOps                  12
AI_ML                    8
Engineering_Systems      7
Cloud                    7
Data Engineering         7
Cybersecurity            6
Tools_Platforms          4
Name: count, dtype: int64


In [23]:
technical_df.head(35 )

,skill,linkedin_frequency,sub_category,confidence
0,data analysis,81964,Data Analytics,1.00
1,excel,41498,Data Analytics,1.00
2,reporting,36528,Data Analytics,1.00
3,quality assurance,36279,Software Engineering,1.00
4,microsoft excel,25773,Data Analytics,1.00
5,python,25168,Programming,1.00
6,sql,22035,Database,1.00
7,electrical engineering,16807,Engineering_Systems,1.00
8,financial reporting,16277,Data Analytics,0.95
9,mechanical engineering,15372,Engineering_Systems,1.00


In [24]:
import pandas as pd

df = pd.read_csv('../Generated Datasets/technical_skill_master.csv')

remove_skills = [
    'reporting',
    'financial reporting',
    'incident reporting',
    'data reporting',
    'sales reporting',
    'project reporting',
    'laboratory testing',
    'drug testing',
    'diagnostic testing',
    'proficiency testing',
    'h&r block income tax course',
    'r&d',
    'business analysis',
    'business analytics'
]

df = df[~df['skill'].str.lower().isin(remove_skills)]

df = df.sort_values(
    'linkedin_frequency',
    ascending=False
).reset_index(drop=True)

df.to_csv(
    '../Generated Datasets/technical_skill_master.csv',
    index=False
)

print("Final Technical Skills:", len(df))
print(df.head())

Final Technical Skills: 114
               skill  linkedin_frequency          sub_category  confidence
0      data analysis               81964        Data Analytics         1.0
1              excel               41498        Data Analytics         1.0
2  quality assurance               36279  Software Engineering         1.0
3    microsoft excel               25773        Data Analytics         1.0
4             python               25168           Programming         1.0


In [25]:
check_terms = [
    'report',
    'reporting',
    'analysis',
    'analytics',
    'testing'
]

for term in check_terms:
    temp = df[df['skill'].str.contains(term, case=False, na=False)]
    if len(temp):
        print(f"\n--- {term.upper()} ---")
        print(temp['skill'].tolist())


--- ANALYSIS ---
['data analysis', 'statistical analysis']

--- ANALYTICS ---
['data analytics']

--- TESTING ---
['testing', 'unit testing', 'software testing', 'integration testing', 'system testing', 'a/b testing', 'performance testing', 'penetration testing']
